# 05 · S_veg swap — production NDVI/VCI vs a DMP anomaly

**Question.** Notebook 04 showed DMP out-ranks the CPI yield everywhere — but DMP measures the OUTCOME, not the cause, so it cannot replace `S_water`/`S_heat`. The term CPI already devotes to observed greenness is `S_veg`. Does sourcing that term from DMP beat the production NDVI/VCI?

**How to run:** put the `planting_pipeline` folder on your Google Drive, run top-to-bottom, approve the Drive-mount and Earth-Engine prompts. Export cells start GEE tasks and return immediately; the scoring cells read the CSVs once the tasks finish (watch https://code.earthengine.google.com/tasks).

## Setup

### Stage 0 · Runtime

Installs the Earth Engine client, geemap, pandas, geopandas and scipy. `scipy` is the one that matters
here: every score in this notebook is a leave-one-out cross-validation with a paired bootstrap interval,
and both come from scipy.

**Expected output.** `installed.`

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas scipy 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine

`EE ready: ok`. The export cells below submit **batch tasks** and return immediately; the scoring cells
read the resulting CSVs. Between the two you have to wait, and you can close the browser while you do.
Watch the queue at code.earthengine.google.com/tasks.

**The export queue is per cloud project.** `ee-manzikye` has left batches in READY for hours. If the
tasks are not entering RUNNING within about 20 minutes, switch `PROJECT` to
`indigo-proxy-484220-q8` and resubmit rather than waiting.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Drive

`pipeline on path: ...`. The scoring scripts read and write `Cropyield-Data/` inside this folder, so the
notebook must `chdir` here for the relative paths to resolve.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Scoring convention used throughout
`n` is small (6–81 zones) in every test here, and a single 70/30 split at n≈45 has a **±0.10 t/ha standard deviation — larger than any effect measured**. So every test below uses **leave-one-out CV** (each free parameter refit on n−1) plus a **paired bootstrap** CI, and reports **Spearman** alongside MAE because rank skill is invariant to the yield ceiling Ym. Reporting a single split would have produced two false positives in this round.

## Design
`S_water` and `S_heat` are computed **once and shared**, so the arms differ in exactly one term:

| arm | S_veg source |
|---|---|
| V | `cpi.s_veg(source='ndvi')` — production MOD13Q1 VCI |
| D | `dmp_yield.s_veg_dmp(mode='vci')` — same Kogan VCI form, DMP source |

Three choices that keep it honest:
1. **The DMP term is an anomaly, not raw biomass.** Raw DMP over-predicted the Kitui wards 23.5x because mixed pixels carry non-crop biomass — but that contamination is largely *static* per pixel, so differencing against the pixel's own DMP climatology cancels it.
2. **Same `VEG_W` = 0.4.** Only the source changes. This also caps the achievable effect: `S_veg` is deliberately down-weighted as a confirmation on water/heat.
3. **Matched climatology (2015–2023) for both arms.** Production `s_veg` defaults to 2003–2024, so arm V here is *not* byte-identical to the shipped product — but an unmatched VCI min/max range would confound source with sample length.

> **Two silent bugs this test surfaced — both produced plausible wrong numbers.**
> 1. Mapped images dropped `system:time_start`, so the inner `filterDate` matched nothing and the seasonal sum returned a **zero-band image**.
> 2. **MOD17A2H v061 in GEE covers only 2021–2026** — every year of the intended climatology was empty. Switched to **MOD17A2HGF** (gap-filled, 2000–2025); gap-filling is an advantage for a climatology, since missing composites would bias min/max.

### Stage 1 · Export the two arms

**The design keeps one thing different and everything else identical.** $S_{\text{water}}$ and
$S_{\text{heat}}$ are computed **once and shared**, so the arms differ in exactly one term:

| Arm | $S_{\text{veg}}$ source |
|---|---|
| V | `cpi.s_veg(source='ndvi')`, the production MOD13Q1 VCI |
| D | `dmp_yield.s_veg_dmp(mode='vci')`, the same Kogan VCI form computed from DMP |

Three choices keep it honest. The DMP term is an **anomaly, not raw biomass**, which cancels the static
mixed-pixel contamination that over-predicted the Kitui wards 23.5-fold in notebook 04. Both arms keep
the same `VEG_W = 0.4`, so only the source changes, which also **caps the achievable effect**, since
$S_{\text{veg}}$ is deliberately down-weighted to a confirmation. Both use a matched 2015 to 2023
climatology, which means arm V is not byte-identical to the shipped product but is comparable to arm D.

**Two silent bugs this test surfaced, both of which produced plausible wrong numbers.**

1. Mapped images dropped `system:time_start`, so the inner `filterDate` matched nothing and the seasonal
   sum returned a **zero-band image**. It did not error; it returned zeros.
2. **MOD17A2H v061 in Earth Engine covers only 2021 to 2026**, so every year of the intended climatology
   was empty. Switched to **MOD17A2HGF**, gap-filled, 2000 to 2025. Gap-filling is an advantage for a
   climatology, because missing composites would bias the minimum and maximum that VCI is built from.

Both are worth remembering generally: in Earth Engine a wrong filter returns an empty collection, and an
empty collection reduces to zeros rather than to an error.

In [ ]:
!python sveg_swap_run.py --variant ke_long
!python sveg_swap_run.py --variant ke_short
!python sveg_swap_run.py --variant et_meher

### Stage 2 · Score

**Expected values.**

| Variant | n | V Spearman | D Spearman | Δ LOO-MAE | Interval | D better in | Fixed-Ym winner |
|---|---|---|---|---|---|---|---|
| Kenya long 2024 | 43 | +0.737 | +0.754 | +0.014 | [−0.024, +0.054] | 77 % | V |
| Kenya short 2024 | 46 | +0.594 | +0.617 | +0.010 | [−0.008, +0.031] | 84 % | V |
| Ethiopia Meher 2024 | 5 | **+0.100** | **+0.700** | +0.075 | [−0.032, +0.185] | 92 % | D |

**The verdict is suggestive, not conclusive.** No variant is statistically significant; every interval
spans zero. But the direction is consistent: DMP ranks better in three of three, and wins 77, 84 and
92 % of resamples across three independent datasets.

**Do not read the fixed-Ym column as a skill statement.** DMP reads **less** stress than NDVI in Kenya,
0.084 against 0.120 in the long rains, so CPI rises and the bias worsens; it reads **more** in Ethiopia,
0.192 against 0.135, so CPI falls and the known over-prediction improves. That column only records
whether the level shift happened to point in a helpful direction, which is not evidence about the
vegetation term.

**Why the swap was not adopted.** The Ethiopia result rests on **n = 5**. Re-run at admin level 2 with
n = 39 it does not hold, so the headline number here has no support at a usable sample size. The
production $S_{\text{veg}}$ stays on MOD13Q1 VCI.

In [ ]:
!python sveg_swap_score.py

## Result

| variant | n | V Spearman | D Spearman | Δ LOO-MAE | CI | D better | fixed-Ym winner |
|---|---|---|---|---|---|---|---|
| KE Long 2024 | 43 | +0.737 | +0.754 | +0.014 | [−0.024,+0.054] | 77% | **V** |
| KE Short 2024 | 46 | +0.594 | +0.617 | +0.010 | [−0.008,+0.031] | 84% | **V** |
| ET Meher 2024 | 5 | **+0.100** | **+0.700** | +0.075 | [−0.032,+0.185] | 92% | **D** |

**No variant is statistically significant** — every CI spans zero. But the direction is consistent: DMP ranks better in 3/3 and wins 77/84/92% of resamples across three independent datasets. Suggestive, not conclusive.

**The fixed-Ym column is not a skill statement.** DMP reads *less* stress than NDVI in Kenya (0.084 vs 0.120 long) so CPI rises and bias worsens; it reads *more* in Ethiopia (0.192 vs 0.135) so CPI falls and the known over-prediction improves. That is just whether the shift happens to point at the existing bias — Ym-dependent. **Spearman is the skill statement.**

### Verdict
**Do not swap for Kenya.** Both seasons show negligible rank gain and a worse-centred CPI under the shipped Ym.

**Ethiopia is a strong lead that n=5 cannot settle.** Spearman 0.10 → 0.70 is the largest skill jump seen in this whole round, and under the production Ym MAE falls 1.586 → 1.268. But at n=5 one region changing position moves the rank correlation that far. **Next step: re-run Ethiopia at admin-2 (zones) to get n into the dozens** — that is the only way to settle it.